In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import LSTM, Dense, Embedding
from keras.optimizers import Adam
from transformers import BertTokenizer, TFBertForSequenceClassification
import tensorflow as tf

In [ ]:
pip install keras

In [ ]:

# Download required NLTK datasets
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Load the datasets
fake_df = pd.read_csv("Fake.csv")
true_df = pd.read_csv("True.csv")

# Add labels
fake_df['label'] = 0  # Fake news
true_df['label'] = 1  # True news

# Combine the datasets
df = pd.concat([fake_df, true_df], ignore_index=True)
df = df.sample(frac=1).reset_index(drop=True)  # Shuffle the data

# Split the data
X = df['title']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize NLTK tools
ps = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Define preprocessing functions
def preprocess_stemming(text):
    text = re.sub('[^a-zA-Z]', ' ', text).lower()
    tokens = [ps.stem(word) for word in text.split() if word not in stop_words]
    return ' '.join(tokens)

def preprocess_lemmatization(text):
    text = re.sub('[^a-zA-Z]', ' ', text).lower()
    tokens = [lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words]
    return ' '.join(tokens)

def preprocess_naive(text):
    return ' '.join([word for word in text.split() if word not in stop_words])

# Define a function to train and evaluate Logistic Regression
def logistic_regression(X_train, X_test, y_train, y_test, vectorizer=None):
    # Fit the vectorizer only on training data if not provided
    if vectorizer is None:
        vectorizer = CountVectorizer()
        X_train_vec = vectorizer.fit_transform(X_train)
    else:
        X_train_vec = vectorizer.transform(X_train)

    # Transform the test data using the same vectorizer
    X_test_vec = vectorizer.transform(X_test)

    # Train the Logistic Regression model
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_vec, y_train)

    # Make predictions and evaluate
    y_pred = model.predict(X_test_vec)
    return accuracy_score(y_test, y_pred), vectorizer


# Define a function to train and evaluate LSTM
def lstm_model(X_train, X_test, y_train, y_test):
    tokenizer = Tokenizer(num_words=5000)
    tokenizer.fit_on_texts(X_train)
    X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=100)
    X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=100)

    model = Sequential()
    model.add(Embedding(input_dim=5000, output_dim=64, input_length=100))
    model.add(LSTM(64))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model.fit(X_train_seq, y_train, epochs=15, batch_size=64, validation_split=0.2, verbose=1)
    loss, accuracy = model.evaluate(X_test_seq, y_test, verbose=0)
    return accuracy

# Define a function to train and evaluate BERT
def bert_model(X_train, X_test, y_train, y_test):
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = TFBertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

    def encode_texts(texts):
        return tokenizer(texts.tolist(), max_length=100, padding=True, truncation=True, return_tensors='tf')

    train_encodings = encode_texts(X_train)
    test_encodings = encode_texts(X_test)

    # Use TensorFlow's Keras optimizer
    model.compile(optimizer="adam", loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(train_encodings['input_ids'], y_train, epochs=15, batch_size=16)
    loss, accuracy = model.evaluate(test_encodings['input_ids'], y_test, verbose=0)
    return accuracy


# Define preprocessing and model combinations
preprocess_methods = {
    'Stemming': preprocess_stemming,
    'Lemmatization': preprocess_lemmatization,
    'Naive': preprocess_naive
}

models = {
    'Logistic Regression': logistic_regression,
    'LSTM': lstm_model
    # 'BERT': bert_model
}

# Evaluate each combination and store results
results = {}
X_train
# for preprocess_name, preprocess_fn in preprocess_methods.items():
#     X_train_prep = X_train.apply(preprocess_fn)
#     X_test_prep = X_test.apply(preprocess_fn)
#     for model_name, model_fn in models.items():
#         accuracy = model_fn(X_train_prep, X_test_prep, y_train, y_test)
#         results[(preprocess_name, model_name)] = accuracy
#         print(f"{preprocess_name} + {model_name}: {accuracy:.4f}")

# # Display the results
# results_df = pd.DataFrame.from_dict(results, orient='index', columns=['Accuracy'])
# print("\nComparison of Preprocessing Methods and Models:")
# print(results_df.sort_values(by='Accuracy', ascending=False))


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,title
36335,"Illinois Republican lawmaker resigns, cites Fa..."
12384,Former head of Muslim Brotherhood dies in hosp...
24419,House to vote on tax bill Tuesday afternoon: a...
24740,Trump looking at fast ways to quit global clim...
27039,YIKES! ACADEMY AWARD WINNING Actor Starring In...
...,...
11284,SUNDAY SCREENING: ‘The Clinton Chronicles’ (1994)
44732,STRANGE: Hillary Goes Off the Rails in Labor U...
38158,"Brazil's new top prosecutor is sworn in, says ..."
860,Economic espionage a 'tremendous problem': U.S...


In [ ]:
df_other=pd.read_csv("train.csv")
df_other = df_other.dropna()
X_test_other = df_other['title']
y_test_other = df_other['label']
# Train the model on the original dataset
train_accuracy = lstm_model(X_train, X_test, y_train, y_test)
print(f"Training Accuracy: {train_accuracy:.4f}")

# Preprocess the new test dataset
x_test_other = X_test_other.apply(preprocess_naive)

# Test the model on a new dataset using the same vectorizer
test_accuracy = lstm_model(X, x_test_other, y, y_test_other)
print(f"Test Accuracy on new dataset: {test_accuracy:.4f}")



Epoch 1/15


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


449/449 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.8797 - loss: 0.3001 - val_accuracy: 0.9598 - val_loss: 0.1116
Epoch 2/15
449/449 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.9760 - loss: 0.0699 - val_accuracy: 0.9692 - val_loss: 0.0794
Epoch 3/15
449/449 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9883 - loss: 0.0382 - val_accuracy: 0.9708 - val_loss: 0.0822
Epoch 4/15
449/449 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9922 - loss: 0.0264 - val_accuracy: 0.9695 - val_loss: 0.0941
Epoch 5/15
449/449 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9958 - loss: 0.0150 - val_accuracy: 0.9659 - val_loss: 0.1231
Epoch 6/15
449/449 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9977 - loss: 0.0081 - val_accuracy: 0.9673 - val_loss: 0.1305
Epoch 7/15
449/449 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9979 - loss: 0.0070 - val_accuracy: 0.9644 - val_loss: 0.1570
Epoch 8/15
449/449 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.9987 - loss: 0.0043 - val_accuracy: 0.9670 - v

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding
from tensorflow.keras.optimizers import Adam

# Initialize NLTK tools
nltk.download('stopwords')
nltk.download('wordnet')
stop_words = set(stopwords.words('english'))
fake_df = pd.read_csv("Fake.csv")
true_df = pd.read_csv("True.csv")

# Add labels
fake_df['label'] = 0  # Fake news
true_df['label'] = 1  # True news

# Combine the datasets
df = pd.concat([fake_df, true_df], ignore_index=True)
df = df.sample(frac=1).reset_index(drop=True)  # Shuffle the data

# Split the data
X = df['title']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

ps = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Define preprocessing functions
def preprocess_stemming(text):
    text = re.sub('[^a-zA-Z]', ' ', text).lower()
    tokens = [ps.stem(word) for word in text.split() if word not in stop_words]
    return ' '.join(tokens)

def preprocess_lemmatization(text):
    text = re.sub('[^a-zA-Z]', ' ', text).lower()
    tokens = [lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words]
    return ' '.join(tokens)

def preprocess_naive(text):
    return ' '.join([word for word in text.split() if word not in stop_words])

# Define classical ML model functions
def logistic_regression(X_train, X_test, y_train, y_test, vectorizer=None):
    if vectorizer is None:
        vectorizer = CountVectorizer()
        X_train_vec = vectorizer.fit_transform(X_train)
    else:
        X_train_vec = vectorizer.transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_vec, y_train)
    y_pred = model.predict(X_test_vec)
    return accuracy_score(y_test, y_pred), vectorizer

def decision_tree(X_train, X_test, y_train, y_test, vectorizer=None):
    if vectorizer is None:
        vectorizer = CountVectorizer()
        X_train_vec = vectorizer.fit_transform(X_train)
    else:
        X_train_vec = vectorizer.transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    model = DecisionTreeClassifier()
    model.fit(X_train_vec, y_train)
    y_pred = model.predict(X_test_vec)
    return accuracy_score(y_test, y_pred), vectorizer

def random_forest(X_train, X_test, y_train, y_test, vectorizer=None):
    if vectorizer is None:
        vectorizer = CountVectorizer()
        X_train_vec = vectorizer.fit_transform(X_train)
    else:
        X_train_vec = vectorizer.transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    model = RandomForestClassifier()
    model.fit(X_train_vec, y_train)
    y_pred = model.predict(X_test_vec)
    return accuracy_score(y_test, y_pred), vectorizer

def gradient_boosting(X_train, X_test, y_train, y_test, vectorizer=None):
    if vectorizer is None:
        vectorizer = CountVectorizer()
        X_train_vec = vectorizer.fit_transform(X_train)
    else:
        X_train_vec = vectorizer.transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    model = GradientBoostingClassifier()
    model.fit(X_train_vec, y_train)
    y_pred = model.predict(X_test_vec)
    return accuracy_score(y_test, y_pred), vectorizer

# Define a function to train and evaluate LSTM
def lstm_model(X_train, X_test, y_train, y_test):
    tokenizer = Tokenizer(num_words=5000)
    tokenizer.fit_on_texts(X_train)
    X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=100)
    X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=100)

    model = Sequential()
    model.add(Embedding(input_dim=5000, output_dim=64, input_length=100))
    model.add(LSTM(64))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
    model.fit(X_train_seq, y_train, epochs=15, batch_size=64, validation_split=0.2, verbose=1)
    loss, accuracy = model.evaluate(X_test_seq, y_test, verbose=0)
    return accuracy
from sklearn.svm import SVC

# Define a function to train and evaluate SVC
def svc_model(X_train, X_test, y_train, y_test, vectorizer=None):
    # Fit the vectorizer only on training data if not provided
    if vectorizer is None:
        vectorizer = CountVectorizer()
        X_train_vec = vectorizer.fit_transform(X_train)
    else:
        X_train_vec = vectorizer.transform(X_train)

    # Transform the test data using the same vectorizer
    X_test_vec = vectorizer.transform(X_test)

    # Initialize and train the SVC model
    model = SVC(kernel='linear', probability=True)
    model.fit(X_train_vec, y_train)

    # Make predictions and evaluate
    y_pred = model.predict(X_test_vec)
    accuracy = accuracy_score(y_test, y_pred)

    # Return both accuracy and the fitted vectorizer
    return accuracy, vectorizer
# Define preprocessing and model combinations
preprocess_methods = {
    'Stemming': preprocess_stemming,
    'Lemmatization': preprocess_lemmatization,
    'Naive': preprocess_naive
}

models = {
    # 'Logistic Regression': logistic_regression,
    # 'Decision Tree': decision_tree,
    # 'Random Forest': random_forest,
    # 'Gradient Boosting': gradient_boosting,
    # 'LSTM': lstm_model
     'SVC': svc_model
}

# Evaluate each combination and store results
results = {}
for preprocess_name, preprocess_fn in preprocess_methods.items():
    X_train_prep = X_train.apply(preprocess_fn)
    X_test_prep = X_test.apply(preprocess_fn)
    for model_name, model_fn in models.items():
        if model_name == 'LSTM':
            accuracy = model_fn(X_train_prep, X_test_prep, y_train, y_test)
        else:
            accuracy, _ = model_fn(X_train_prep, X_test_prep, y_train, y_test)
        results[(preprocess_name, model_name)] = accuracy
        print(f"{preprocess_name} + {model_name}: {accuracy:.4f}")

# Display the results
results_df = pd.DataFrame.from_dict(results, orient='index', columns=['Accuracy'])
print("\nComparison of Preprocessing Methods and Models:")
print(results_df.sort_values(by='Accuracy', ascending=False))


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Stemming + SVC: 0.9481
Lemmatization + SVC: 0.9476
Naive + SVC: 0.9837

Comparison of Preprocessing Methods and Models:
                      Accuracy
(Naive, SVC)          0.983742
(Stemming, SVC)       0.948107
(Lemmatization, SVC)  0.947550


In [ ]:
import joblib
# Save trained models and vectorizer
for model_name, model_fn in models.items():
    if model_name == 'LSTM':
        continue  # Skipping LSTM as it requires a different saving approach
        # Save the LSTM model
        lstm_model_instance.save("LSTM_model.h5")

        # Load the LSTM model
        from tensorflow.keras.models import load_model
        lstm_model_instance = load_model("LSTM_model.h5")

    else:
        accuracy, vectorizer = model_fn(X_train_prep, X_test_prep, y_train, y_test)
        # Save the model
        joblib.dump(model_fn, f"{model_name}_model.pkl")
        # Save the vectorizer
        joblib.dump(vectorizer, f"{model_name}_vectorizer.pkl")


In [ ]:
def demo_prediction(title):
    # List of model names to load
    model_names = ['Logistic Regression', 'Decision Tree', 'Random Forest', 'Gradient Boosting', 'SVC']
    predictions = {}

    for model_name in model_names:
        # Load the model and vectorizer
        model = joblib.load(f"{model_name}_model.pkl")
        vectorizer = joblib.load(f"{model_name}_vectorizer.pkl")

        # Preprocess the input title (using naive preprocessing as an example)
        preprocessed_title = preprocess_naive(title)

        # Vectorize the input title
        title_vec = vectorizer.transform([preprocessed_title])
        # Load the LSTM tokenizer and model
        tokenizer = joblib.load("lstm_tokenizer.pkl")
        lstm_model_instance = load_model("LSTM_model.h5")

        # Preprocess and tokenize the input title
        title_seq = pad_sequences(tokenizer.texts_to_sequences([preprocessed_title]), maxlen=100)

        # Make prediction
        lstm_prediction = lstm_model_instance.predict(title_seq)[0][0]
        predictions['LSTM'] = "Fake" if lstm_prediction < 0.5 else "True"
        # Make prediction
        prediction = model.predict(title_vec)[0]
        predictions[model_name] = "Fake" if prediction == 0 else "True"

    # Print all predictions
    print("\nPredictions for the input title:")
    for model_name, prediction in predictions.items():
        print(f"{model_name}: {prediction}")


In [ ]:
# Example title
input_title = "Breaking news: COVID-19 vaccine found to be 95% effective"
demo_prediction(input_title)


In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Embedding
from tensorflow.keras.optimizers import Adam

# Initialize NLTK tools
nltk.download('stopwords')
nltk.download('wordnet')
stop_words = set(stopwords.words('english'))
ps = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Define preprocessing functions
def preprocess_stemming(text):
    text = re.sub('[^a-zA-Z]', ' ', text).lower()
    tokens = [ps.stem(word) for word in text.split() if word not in stop_words]
    return ' '.join(tokens)

def preprocess_lemmatization(text):
    text = re.sub('[^a-zA-Z]', ' ', text).lower()
    tokens = [lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words]
    return ' '.join(tokens)

def preprocess_naive(text):
    return ' '.join([word for word in text.split() if word not in stop_words])

# Define classical ML model functions
def train_and_save_model(model, model_name, X_train, y_train, vectorizer):
    X_train_vec = vectorizer.fit_transform(X_train)
    model.fit(X_train_vec, y_train)
    joblib.dump(model, f"{model_name}_model.pkl")
    joblib.dump(vectorizer, f"{model_name}_vectorizer.pkl")

def logistic_regression(X_train, y_train, vectorizer):
    model = LogisticRegression(max_iter=1000)
    train_and_save_model(model, "Logistic_Regression", X_train, y_train, vectorizer)

def decision_tree(X_train, y_train, vectorizer):
    model = DecisionTreeClassifier()
    train_and_save_model(model, "Decision_Tree", X_train, y_train, vectorizer)

def random_forest(X_train, y_train, vectorizer):
    model = RandomForestClassifier()
    train_and_save_model(model, "Random_Forest", X_train, y_train, vectorizer)

def gradient_boosting(X_train, y_train, vectorizer):
    model = GradientBoostingClassifier()
    train_and_save_model(model, "Gradient_Boosting", X_train, y_train, vectorizer)

def svc_model(X_train, y_train, vectorizer):
    model = SVC(kernel='linear', probability=True)
    train_and_save_model(model, "SVC", X_train, y_train, vectorizer)

# Define a function to train and save LSTM
def train_lstm(X_train, y_train):
    tokenizer = Tokenizer(num_words=5000)
    tokenizer.fit_on_texts(X_train)
    X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=100)

    model = Sequential()
    model.add(Embedding(input_dim=5000, output_dim=64, input_length=100))
    model.add(LSTM(64))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
    model.fit(X_train_seq, y_train, epochs=5, batch_size=64, validation_split=0.2, verbose=1)

    model.save("LSTM_model.h5")
    joblib.dump(tokenizer, "lstm_tokenizer.pkl")

# Train and save all models
def train_all_models(X_train, y_train):
    vectorizer = CountVectorizer()
    logistic_regression(X_train, y_train, vectorizer)
    decision_tree(X_train, y_train, vectorizer)
    random_forest(X_train, y_train, vectorizer)
    gradient_boosting(X_train, y_train, vectorizer)
    svc_model(X_train, y_train, vectorizer)
    train_lstm(X_train, y_train)

# Demo function for making predictions
def demo_prediction(title):
    preprocess_fn = preprocess_naive  # Using naive preprocessing for demo
    preprocessed_title = preprocess_fn(title)

    model_names = ["Logistic_Regression", "Decision_Tree", "Random_Forest", "Gradient_Boosting", "SVC"]
    predictions = {}

    for model_name in model_names:
        model = joblib.load(f"{model_name}_model.pkl")
        vectorizer = joblib.load(f"{model_name}_vectorizer.pkl")
        title_vec = vectorizer.transform([preprocessed_title])
        prediction = model.predict(title_vec)[0]
        predictions[model_name] = "Fake" if prediction == 0 else "True"

    # LSTM model prediction
    tokenizer = joblib.load("lstm_tokenizer.pkl")
    lstm_model_instance = load_model("LSTM_model.h5")
    title_seq = pad_sequences(tokenizer.texts_to_sequences([preprocessed_title]), maxlen=100)
    lstm_prediction = lstm_model_instance.predict(title_seq)[0][0]
    predictions["LSTM"] = "Fake" if lstm_prediction < 0.5 else "True"

    # Print predictions
    print("\nPredictions for the input title:")
    for model_name, prediction in predictions.items():
        print(f"{model_name}: {prediction}")

# Example Usage
if __name__ == "__main__":
    # Load your dataset
    fake_df = pd.read_csv("Fake.csv")
    true_df = pd.read_csv("True.csv")

    # Add labels and combine datasets
    fake_df['label'] = 0
    true_df['label'] = 1
    df = pd.concat([fake_df, true_df], ignore_index=True)
    df = df.sample(frac=1).reset_index(drop=True)

    # Split the data
    X = df['title']
    y = df['label']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Train all models
    print("Training all models...")
    train_all_models(X_train, y_train)

    # Demo prediction
    input_title = "Trump killed his wife and his children"
    demo_prediction(input_title)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Training all models...
Epoch 1/5


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


165/449 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8227 - loss: 0.4320

KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from nltk.corpus import stopwords
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Embedding
from tensorflow.keras.optimizers import Adam

# Initialize NLTK tools
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Define Naive preprocessing function
def preprocess_naive(text):
    return ' '.join([word for word in text.split() if word not in stop_words])

# Define classical ML model training and saving function
def train_and_save_model(model, model_name, X_train, y_train):
    # Create a pipeline with CountVectorizer and the model
    pipeline = Pipeline([
        ('vectorizer', CountVectorizer()),
        ('classifier', model)
    ])
    # Train the model
    pipeline.fit(X_train, y_train)
    # Save the pipeline (model + vectorizer)
    joblib.dump(pipeline, f"{model_name}_naive.pkl")
    print(f"Saved {model_name} model.")

# Define a function to train and save LSTM
def train_lstm(X_train, y_train):
    tokenizer = Tokenizer(num_words=5000)
    tokenizer.fit_on_texts(X_train)
    X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=100)

    model = Sequential()
    model.add(Embedding(input_dim=5000, output_dim=64, input_length=100))
    model.add(LSTM(64))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
    model.fit(X_train_seq, y_train, epochs=5, batch_size=64, validation_split=0.2, verbose=1)

    # Save the model and tokenizer
    model.save("LSTM_naive.h5")
    joblib.dump(tokenizer, "lstm_tokenizer_naive.pkl")
    print("Saved LSTM model.")

# Train all models with Naive preprocessing
def train_all_models(X_train, y_train):
    # Apply Naive preprocessing
    X_train_prep = X_train.apply(preprocess_naive)

    # Train and save classical ML models
    train_and_save_model(LogisticRegression(max_iter=1000), "Logistic_Regression", X_train_prep, y_train)
    train_and_save_model(DecisionTreeClassifier(), "Decision_Tree", X_train_prep, y_train)
    train_and_save_model(RandomForestClassifier(), "Random_Forest", X_train_prep, y_train)
    train_and_save_model(GradientBoostingClassifier(), "Gradient_Boosting", X_train_prep, y_train)
    train_and_save_model(SVC(kernel='linear', probability=True), "SVC", X_train_prep, y_train)

    # Train and save LSTM model
    train_lstm(X_train_prep, y_train)

# Demo function for making predictions
def demo_prediction(title):
    # Preprocess the input title using Naive method
    preprocessed_title = preprocess_naive(title)

    # List of model names
    model_names = ["Logistic_Regression", "Decision_Tree", "Random_Forest", "Gradient_Boosting", "SVC"]
    predictions = {}

    # Make predictions using classical ML models
    for model_name in model_names:
        model = joblib.load(f"{model_name}_naive.pkl")
        prediction = model.predict([preprocessed_title])[0]
        predictions[model_name] = "Fake" if prediction == 0 else "True"

    # Make prediction using LSTM model
    tokenizer = joblib.load("lstm_tokenizer_naive.pkl")
    lstm_model_instance = load_model("LSTM_naive.h5")
    title_seq = pad_sequences(tokenizer.texts_to_sequences([preprocessed_title]), maxlen=100)
    lstm_prediction = lstm_model_instance.predict(title_seq)[0][0]
    predictions["LSTM"] = "Fake" if lstm_prediction < 0.5 else "True"

    # Print predictions
    print("\nPredictions for the input title:")
    for model_name, prediction in predictions.items():
        print(f"{model_name}: {prediction}")

# Example Usage
if __name__ == "__main__":
    # Load your dataset
    fake_df = pd.read_csv("Fake.csv")
    true_df = pd.read_csv("True.csv")

    # Add labels and combine datasets
    fake_df['label'] = 0
    true_df['label'] = 1
    df = pd.concat([fake_df, true_df], ignore_index=True)
    df = df.sample(frac=1).reset_index(drop=True)

    # Split the data
    X = df['title']
    y = df['label']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Train all models
    print("Training all models with Naive preprocessing...")
    train_all_models(X_train, y_train)

    # Demo prediction
    input_title = "Breaking news: COVID-19 vaccine found to be 95% effective"
    demo_prediction(input_title)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Training all models with Naive preprocessing...
Saved Logistic_Regression model.
Saved Decision_Tree model.
Saved Random_Forest model.
Saved Gradient_Boosting model.
Saved SVC model.
Epoch 1/5


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


449/449 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.9008 - loss: 0.2269 - val_accuracy: 0.9871 - val_loss: 0.0392
Epoch 2/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9925 - loss: 0.0209 - val_accuracy: 0.9875 - val_loss: 0.0375
Epoch 3/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.9971 - loss: 0.0097 - val_accuracy: 0.9854 - val_loss: 0.0472
Epoch 4/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.9979 - loss: 0.0062 - val_accuracy: 0.9820 - val_loss: 0.0757
Epoch 5/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.9991 - loss: 0.0027 - val_accuracy: 0.9855 - val_loss: 0.0624


Saved LSTM model.


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step

Predictions for the input title:
Logistic_Regression: Fake
Decision_Tree: Fake
Random_Forest: Fake
Gradient_Boosting: True
SVC: True
LSTM: Fake


In [ ]:
input_title = "Trump killed his children and wife"
demo_prediction(input_title)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step

Predictions for the input title:
Logistic_Regression: True
Decision_Tree: True
Random_Forest: True
Gradient_Boosting: True
SVC: True
LSTM: True


In [ ]:
input_title = 'First German immigration law on agenda as Merkel seeks coalition'
demo_prediction(input_title)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step

Predictions for the input title:
Logistic_Regression: True
Decision_Tree: True
Random_Forest: True
Gradient_Boosting: True
SVC: True
LSTM: True


In [ ]:
X[0]


'First German immigration law on agenda as Merkel seeks coalition'

In [ ]:
import pandas as pd
import numpy as np
import re
import joblib
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import load_model

# Naive preprocessing function (removes stopwords and lowercases text)
def preprocess_naive(text):
    text = re.sub('[^a-zA-Z]', ' ', text).lower()
    return ' '.join(text.split())

# Function to train and save LSTM model
def train_lstm(X_train, y_train):

    tokenizer = Tokenizer(num_words=5000)
    tokenizer.fit_on_texts(X_train)
    X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=100)

    model = Sequential()
    model.add(Embedding(input_dim=5000, output_dim=64, input_length=100))
    model.add(LSTM(64))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
    model.fit(X_train_seq, y_train, epochs=10, batch_size=64, validation_split=0.2, verbose=1)

    model.save("LSTM_model.h5")
    joblib.dump(tokenizer, "lstm_tokenizer.pkl")

# Demo prediction function
def demo_prediction(title):
    preprocessed_title = preprocess_naive(title)

    # Load the tokenizer and model
    tokenizer = joblib.load("lstm_tokenizer.pkl")
    lstm_model_instance = load_model("LSTM_model.h5")

    # Preprocess and predict
    title_seq = pad_sequences(tokenizer.texts_to_sequences([preprocessed_title]), maxlen=100)
    lstm_prediction = lstm_model_instance.predict(title_seq)[0][0]

    # Print the prediction result
    prediction = "Fake" if lstm_prediction < 0.5 else "True"
    print(f"Prediction for the input title: {prediction}")

# Main script to train and demonstrate the model
if __name__ == "__main__":
    # Load the dataset
    fake_df = pd.read_csv("Fake.csv")
    true_df = pd.read_csv("True.csv")

    # Add labels and combine datasets
    fake_df['label'] = 0
    true_df['label'] = 1
    df = pd.concat([fake_df, true_df], ignore_index=True)
    df = df.sample(frac=1).reset_index(drop=True)

    # Split the data
    X = df['title'].apply(preprocess_naive)
    y = df['label']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Train the LSTM model
    print("Training LSTM model...")
    train_lstm(X_train, y_train)

    # Demo prediction
    input_title = "Trump killed his wife and his children"
    demo_prediction(input_title)


Training LSTM model...
Epoch 1/10


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


449/449 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step - accuracy: 0.8859 - loss: 0.2958 - val_accuracy: 0.9609 - val_loss: 0.1060
Epoch 2/10
449/449 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.9765 - loss: 0.0660 - val_accuracy: 0.9649 - val_loss: 0.0960
Epoch 3/10
449/449 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9864 - loss: 0.0415 - val_accuracy: 0.9662 - val_loss: 0.0998
Epoch 4/10
449/449 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.9922 - loss: 0.0245 - val_accuracy: 0.9633 - val_loss: 0.1240
Epoch 5/10
449/449 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9959 - loss: 0.0156 - val_accuracy: 0.9623 - val_loss: 0.1388
Epoch 6/10
449/449 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9964 - loss: 0.0107 - val_accuracy: 0.9602 - val_loss: 0.1957
Epoch 7/10
449/449 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9969 - loss: 0.0100 - val_accuracy: 0.9601 - val_loss: 0.1864
Epoch 8/10
449/449 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9991 - loss: 0.0037 - val_accuracy: 0.9580 - v

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step
Prediction for the input title: Fake


In [ ]:
input_title = "World Leaders Seek Stability With China as President Biden Exits the Stage"
demo_prediction(input_title)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 512ms/step
Prediction for the input title: True
